# DRL Examples

Using [stable-baseline3](https://github.com/DLR-RM/stable-baselines3)

In [42]:
%pip install -q stable-baselines3[extra] tensorboard

Note: you may need to restart the kernel to use updated packages.


In [43]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")

## Deep Q-Network (DQN)

In [9]:
import gymnasium as gym

Train the model

In [45]:
from stable_baselines3 import DQN

In [46]:
TRAIN = False
#TRAIN = True

if TRAIN:
    env = gym.make("LunarLander-v3", render_mode=None)

    model = DQN(
        "MlpPolicy",
        env,
        verbose=0,
        exploration_final_eps=0.2,
        target_update_interval=250,
        learning_rate=0.01,
        batch_size=64,
        tensorboard_log = "./tb_logs_LL"
    )

First, I learn with resetting the timesteps.

In [47]:
if TRAIN: 
    model.learn(total_timesteps=10000, log_interval=1, progress_bar=True, reset_num_timesteps=True)

1. Run in the Terminal (you may have to adjust the logdir): `tensorboard --logdir DRL/tb_logs_LL/`
2. Follow the instruction on the terminal to open the tensorboard in the browser.
3. Run the following line a few times as long as you see in improvement in the tensorboard. Learning flattened out at about 60k episodes for me.

In [48]:
if TRAIN: 
    model.learn(total_timesteps=20000, log_interval=1, progress_bar=True, reset_num_timesteps=False)

Now, we can reduce the learning rate and, if necessary the exploration rate.

In [49]:
from stable_baselines3.common.utils import ConstantSchedule

if TRAIN:
    model.learning_rate = .001
    model.exploration_schedule = ConstantSchedule(.1)

Run more training as long as the model improves.

In [50]:
if TRAIN: 
    model.learn(total_timesteps=20000, log_interval=1, progress_bar=True, reset_num_timesteps=False)

Save the final model.

In [51]:
if TRAIN:
    env.close()
    model.save("dqn_lunarlander")

Load and run the model.

In [52]:
model = DQN.load("dqn_lunarlander")

In [53]:
from gymnasium_display_recorder import VideoWrapper, show

env_orig = gym.make('LunarLander-v3', render_mode="rgb_array")
env = VideoWrapper(env_orig, 'DQN', render_fps=30)

obs, info = env.reset()
while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
        obs, info = env.reset()


show(env)
env.close()

Videos already exist, I remove them first!
Showing: ./videos/video_DQN-episode-0.mp4


## Proximal Policy Optimization (PPO)

In [13]:
from stable_baselines3 import PPO

In [29]:
TRAIN = False
#TRAIN = True

if TRAIN:
    env = gym.make("LunarLander-v3", render_mode=None)

    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,
        learning_rate=0.01,
        n_steps=1024,
        batch_size=64,
        n_epochs=4,
        gamma=.99,
        gae_lambda=0.95,
        tensorboard_log = "./tb_logs_LL"
    )


In [ ]:
if TRAIN: 
    model.learn(total_timesteps=10000, log_interval=1, progress_bar=True, reset_num_timesteps=True)

Output()

In [ ]:
if TRAIN: 
    model.learn(total_timesteps=20000, log_interval=1, progress_bar=True, reset_num_timesteps=False)

Output()

AssertionError: 

In [25]:
if TRAIN:
    env.close()
    model.save("ppo_lunarlander")

In [ ]:
model = PPO.load("ppo_lunarlander")

In [ ]:
from gymnasium_display_recorder import VideoWrapper, show

env_orig = gym.make('LunarLander-v3', render_mode="rgb_array")
env = VideoWrapper(env_orig, 'PPO', render_fps=30)

for _ in range(5):
obs, info = env.reset()
    while True:
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break
            obs, info = env.reset()
        

    show(env)

env.close()

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/DRL/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Showing: ./videos/video_PPO-episode-0.mp4
